# Deaths of Despair — Layer 3: Deep Dives

Directed analysis: lag effects, regression, threshold patterns.

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import sys
sys.path.insert(0, "/Users/matt/data-adventures")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir
import statsmodels.api as sm

PROJECT = "deaths-of-despair"
config = load_config(PROJECT)
processed_dir = get_data_processed_dir(config)

state_panel = pd.read_parquet(processed_dir / "state_panel.parquet")
national_panel = pd.read_parquet(processed_dir / "national_panel.parquet")

print(f"state_panel: {state_panel.shape[0]:,} rows | years {int(state_panel.year.min())}–{int(state_panel.year.max())}")


state_panel: 918 rows | years 1999–2016


## Deep-dive 1: Lag analysis — does manufacturing decline precede rising deaths?

In [2]:
# National annual averages for lag analysis
nat = (
    state_panel.groupby("year")[["manufacturing_employees_thousands","deaths_despair_rate","unemployment_rate"]]
    .mean()
    .reset_index()
    .sort_values("year")
)

# Compute % change in manufacturing from prior year (national)
nat["mfg_pct_change"] = nat["manufacturing_employees_thousands"].pct_change() * 100

# Cross-correlate: mfg change lagged 1-10 years vs deaths_despair_rate change
corrs = {}
for lag in range(0, 9):
    shifted_mfg = nat["mfg_pct_change"].shift(lag)
    valid = ~(shifted_mfg.isna() | nat["deaths_despair_rate"].isna())
    if valid.sum() > 5:
        r = np.corrcoef(shifted_mfg[valid], nat["deaths_despair_rate"][valid])[0, 1]
        corrs[lag] = r

print("Lag (years) | Correlation (mfg change vs deaths of despair rate)")
for lag, r in corrs.items():
    bar = "█" * int(abs(r) * 30)
    direction = "neg" if r < 0 else "pos"
    print(f"  Lag {lag:2d}      | r={r:+.3f}  {bar}")

fig = go.Figure()
fig.add_trace(go.Bar(x=list(corrs.keys()), y=list(corrs.values()),
    marker_color=["red" if v < 0 else "blue" for v in corrs.values()]))
fig.update_layout(
    title="Lag Correlation: Manufacturing Job Loss → Deaths of Despair",
    xaxis_title="Lag (years)", yaxis_title="Pearson r",
    shapes=[dict(type="line", x0=-0.5, x1=len(corrs)-0.5, y0=0, y1=0, line_dash="dash")]
)
fig.show()


Lag (years) | Correlation (mfg change vs deaths of despair rate)
  Lag  0      | r=+0.461  █████████████
  Lag  1      | r=+0.464  █████████████
  Lag  2      | r=+0.430  ████████████
  Lag  3      | r=+0.394  ███████████
  Lag  4      | r=+0.348  ██████████
  Lag  5      | r=+0.163  ████
  Lag  6      | r=-0.187  █████
  Lag  7      | r=-0.431  ████████████
  Lag  8      | r=+0.243  ███████


## Deep-dive 2: OLS regression — manufacturing employment predicts despair

In [3]:
# Panel regression: deaths_despair_rate ~ mfg_employees + unemployment
reg_data = state_panel.dropna(
    subset=["deaths_despair_rate","manufacturing_employees_thousands","unemployment_rate"]
).copy()
# Cast to float to avoid statsmodels dtype issues
for col in ["deaths_despair_rate","manufacturing_employees_thousands","unemployment_rate"]:
    reg_data[col] = reg_data[col].astype(float)

X = sm.add_constant(
    reg_data[["manufacturing_employees_thousands","unemployment_rate"]].astype(float)
)
y = reg_data["deaths_despair_rate"].astype(float)
model = sm.OLS(y, X).fit()
print(model.summary())
print()
print(f"R² = {model.rsquared:.3f} | Adj R² = {model.rsquared_adj:.3f}")
print(f"Manufacturing coeff: {model.params['manufacturing_employees_thousands']:.4f}  (p={model.pvalues['manufacturing_employees_thousands']:.4f})")


                             OLS Regression Results                            
Dep. Variable:     deaths_despair_rate   R-squared:                       0.174
Model:                             OLS   Adj. R-squared:                  0.172
Method:                  Least Squares   F-statistic:                     79.45
Date:                 Tue, 03 Mar 2026   Prob (F-statistic):           4.95e-32
Time:                         21:16:52   Log-Likelihood:                -2631.8
No. Observations:                  756   AIC:                             5270.
Df Residuals:                      753   BIC:                             5283.
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

## Deep-dive 3: The three waves — opioid prescription → heroin → fentanyl

In [4]:
nat_sorted = national_panel.dropna(subset=["overdose_death_rate"]).sort_values("year")

# Annotate three eras
era1 = nat_sorted[nat_sorted["year"] <= 2010]
era2 = nat_sorted[(nat_sorted["year"] > 2010) & (nat_sorted["year"] <= 2013)]
era3 = nat_sorted[nat_sorted["year"] > 2013]

fig = go.Figure()
fig.add_trace(go.Scatter(x=era1["year"], y=era1["overdose_death_rate"],
    mode="lines+markers", name="Wave 1: Prescription Opioids",
    line=dict(color="#e67e22", width=3)))
fig.add_trace(go.Scatter(x=era2["year"], y=era2["overdose_death_rate"],
    mode="lines+markers", name="Wave 2: Heroin",
    line=dict(color="#c0392b", width=3)))
fig.add_trace(go.Scatter(x=era3["year"], y=era3["overdose_death_rate"],
    mode="lines+markers", name="Wave 3: Fentanyl",
    line=dict(color="#8e44ad", width=3)))

# Add annotations
fig.add_annotation(x=2005, y=10.4, text="Rx Opioids<br>Flood the Market", showarrow=True, arrowhead=2)
fig.add_annotation(x=2011.5, y=13.5, text="Heroin as<br>Cheaper Alt.", showarrow=True, arrowhead=2)
fig.add_annotation(x=2015, y=18, text="Fentanyl<br>Enters Supply", showarrow=True, arrowhead=2)

fig.update_layout(
    title="The Three Waves of the Opioid Crisis (1999–2016)",
    xaxis_title="Year", yaxis_title="Overdose Deaths per 100k",
    legend=dict(orientation="h", y=1.05),
)
fig.show()


## Deep-dive 4: State-level regression — top drivers

In [5]:
# Compute per-state linear trends using numpy polyfit (avoids statsmodels dtype issues)
state_trends = []
for state in state_panel["state_abbr"].unique():
    df = state_panel[state_panel["state_abbr"] == state].dropna(
        subset=["deaths_despair_rate","year"]
    ).sort_values("year")
    if len(df) < 5:
        continue
    try:
        years = df["year"].astype(float).values
        rates = df["deaths_despair_rate"].astype(float).values
        slope, intercept = np.polyfit(years, rates, 1)
        ss_res = np.sum((rates - (slope * years + intercept))**2)
        ss_tot = np.sum((rates - rates.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        state_trends.append({
            "state_abbr": state,
            "trend_slope": round(slope, 4),
            "trend_r2": round(r2, 3),
            "mean_despair_rate": round(df["deaths_despair_rate"].mean(), 2),
            "region": df["region"].iloc[0] if "region" in df.columns else "Unknown",
        })
    except Exception as exc:
        print(f"  {state}: {exc}")

trends_df = pd.DataFrame(state_trends).sort_values("trend_slope", ascending=False)
print("States with steepest upward trend (deaths per 100k per year):")
print(trends_df.head(10)[["state_abbr","trend_slope","mean_despair_rate","region"]].to_string(index=False))

if not trends_df.empty:
    fig = px.bar(
        trends_df.head(15), x="state_abbr", y="trend_slope",
        color="region",
        title="States with Steepest Increase in Deaths of Despair (annual slope)",
        labels={"trend_slope": "Annual increase (deaths per 100k per year)", "state_abbr": "State"},
    )
    fig.show()


States with steepest upward trend (deaths per 100k per year):
state_abbr  trend_slope  mean_despair_rate                 region
        WV       2.5659              38.40             Appalachia
        NH       1.8673              27.05                  Other
        OH       1.8102              26.70 Rust Belt / Appalachia
        KY       1.7604              32.49             Appalachia
        PA       1.4553              27.84 Rust Belt / Appalachia
        RI       1.4438              25.22                  Other
        WY       1.3995              32.82                  Other
        IN       1.3270              24.88              Rust Belt
        OK       1.3139              31.72                  Other
        MO       1.2632              27.02              Rust Belt
